1. fetch data finance with yfinance then count the bullish and bearish candlestick
2. calculate the payoff matrix with as well as @Latex/PakHeru_Presentasi4/Adv_Gemini_plan until.md we get the probability of each state as well as @Latex/PakGagus_Presentasi3/rencana_presentasi_analisis_numerik.md
3. fetch calculated value of payoff to calculate bias
4. construct the density matrix to get the value of von neumann entropy as interaction (J_ij)
5. fetch the bias and interaction value to hamiltonian ising 
6. use vqe algorithm to get the ground stat with 

In [1]:
import yfinance as yf
import pandas as pd

# Daftar saham yang akan diambil datanya
tickers = ['BBCA.JK', 'TPIA.JK', 'ASII.JK', 'TLKM.JK']
period = '6mo'
interval = '1d'

# Dictionary untuk menyimpan hasil perhitungan
candlestick_counts = {}

print("Fetching data and counting candlesticks...\n")

for ticker in tickers:
    # Mengambil data historis
    df = yf.download(ticker, period=period, interval=interval, progress=False)
    
    if df.empty:
        print(f"No data found for {ticker}")
        continue
    
    # Menangani MultiIndex columns jika yfinance versi terbaru
    if isinstance(df.columns, pd.MultiIndex):
        open_price = df['Open'][ticker]
        close_price = df['Close'][ticker]
    else:
        open_price = df['Open']
        close_price = df['Close']
        
    # Perhitungan Candlestick
    # Bullish: Close > Open
    # Bearish: Close < Open
    # Neutral/Doji: Close == Open
    bullish_count = int((close_price > open_price).sum())
    bearish_count = int((close_price < open_price).sum())
    neutral_count = int((close_price == open_price).sum())
    
    candlestick_counts[ticker] = {
        'Bullish': bullish_count,
        'Bearish': bearish_count,
        'Neutral': neutral_count,
        'Total Data': len(df)
    }
    
    print(f"[{ticker}]")
    print(f"  Bullish : {bullish_count}")
    print(f"  Bearish : {bearish_count}")
    print(f"  Neutral : {neutral_count}")
    print(f"  Total   : {len(df)}\n")

# Menampilkan rekap dalam bentuk DataFrame
summary_df = pd.DataFrame(candlestick_counts).T
summary_df

Fetching data and counting candlesticks...

[BBCA.JK]
  Bullish : 52
  Bearish : 57
  Neutral : 16
  Total   : 125

[TPIA.JK]
  Bullish : 41
  Bearish : 73
  Neutral : 11
  Total   : 125

[ASII.JK]
  Bullish : 52
  Bearish : 61
  Neutral : 12
  Total   : 125

[TLKM.JK]
  Bullish : 56
  Bearish : 64
  Neutral : 5
  Total   : 125



,Bullish,Bearish,Neutral,Total Data
BBCA.JK,52,57,16,125
TPIA.JK,41,73,11,125
ASII.JK,52,61,12,125
TLKM.JK,56,64,5,125


### Contoh Kalkulasi Manual: Payoff & Probabilitas (Metode Markowitz)

Misalkan kita mengamati 100 hari riwayat perdagangan untuk pasangan saham **Leader (BBCA)** dan **Follower (ASII)**.

#### 1. Parameter Pasar (Endogenous Risk Aversion)
Misalkan dari seluruh data historis bursa didapatkan:
- Rata-rata return pasar ($\mu_{avg}$) = `0.001` (0.1%)
- Rata-rata volatilitas pasar ($\sigma_{avg}$) = `0.015` (1.5%)
- Penghitungan $\lambda_{\text{market}}$:
  $$ \lambda = \frac{\sigma_{avg}}{\mu_{avg} + \sigma_{avg}} = \frac{0.015}{0.001 + 0.015} = \frac{0.015}{0.016} \approx 0.9375 $$

#### 2. Perhitungan pada State $|00\rangle$ (Keduanya Sama-Sama Naik, $R_t > 0$)
Misalkan mereka naik secara bersamaan di hari yang sama sebanyak **45 hari** ($n_{00} = 45$).
- **Probabilitas ($P_{00}$)**: $\frac{n_{00}}{n_{total}} = \frac{45}{100} = 0.45$
- **Amplitudo Kuantum ($a_{00}$)**: $\sqrt{P_{00}} = \sqrt{0.45} \approx 0.6708$

#### 3. Perhitungan Utilitas Markowitz (Payoff) pada State $|00\rangle$
Misalkan jika kita ambil subset 45 hari di mana KEDUANYA NAIK tersebut, kita dapati:
- Rata-rata return harian BBCA $\mu_L^{(00)}$ = `0.010`
- Std dev return BBCA $\sigma_L^{(00)}$ = `0.005`
- Rata-rata return harian ASII $\mu_F^{(00)}$ = `0.012`
- Std dev return ASII $\sigma_F^{(00)}$ = `0.008`

**Maka Nilai Payoff BBCA (Leader) di Matriks (Baris 0, Kolom 0):**
$$ U_L^{(00)} = (1 - \lambda) \cdot \mu_L^{(00)} - \lambda \cdot \sigma_L^{(00)} $$
$$ U_L^{(00)} = (1 - 0.9375) \cdot 0.010 - 0.9375 \cdot 0.005 $$
$$ U_L^{(00)} = (0.0625 \cdot 0.010) - 0.0046875 = 0.000625 - 0.0046875 = -0.0040625 $$

**Sedangkan Nilai Payoff ASII (Follower) di Matriks (Baris 0, Kolom 0):**
$$ U_F^{(00)} = (1 - \lambda) \cdot \mu_F^{(00)} - \lambda \cdot \sigma_F^{(00)} $$
$$ U_F^{(00)} = (0.0625 \cdot 0.012) - (0.9375 \cdot 0.008) = 0.00075 - 0.0075 = -0.00675 $$

#### 4. Iterasi Selanjutnya (Penempatan di Matriks 2x2)
Proses (Frekuensi $\rightarrow$ $P_{ij}$ $\rightarrow$ Amplitudo Kuantum $a_{ij}$ $\rightarrow$ Nilai Subset $\mu, \sigma$ $\rightarrow$ $U_L, U_F$) ini kemudian diulangi pada:
- State $|01\rangle$: BBCA Naik, ASII Turun
- State $|10\rangle$: BBCA Turun, ASII Naik
- State $|11\rangle$: Keduanya Sama-Sama Turun

Sehingga menghasilkan secara utuh `Payoff_L` (Leader), `Payoff_F` (Follower), `Probabilities`, dan `Quantum_Coeffs`.


In [2]:
import numpy as np
import itertools

print("Calculating Log Returns and Market Risk Aversion...\n")

# Mengunduh ulang data harga Adjusted Close atau Close karena df sebelumnya tertimpa per iterasi saham tunggal
data = yf.download(tickers, period=period, interval=interval, progress=False)

if isinstance(data.columns, pd.MultiIndex):
    if 'Adj Close' in data.columns.levels[0]:
        prices = data['Adj Close']
    else:
        prices = data['Close']
else:
    prices = data['Close']

prices = prices.dropna()

# 1. Menghitung Daily Log Return: R_t = ln(P_t / P_{t-1})
log_returns = np.log(prices / prices.shift(1)).dropna()

# 2. Diskritisasi State: |0> (Naik, R_t > 0) dan |1> (Turun, R_t <= 0)
states = (log_returns <= 0).astype(int)

# 3. Menghitung lambda (Risk Aversion Endogen) secara keseluruhan pasar
mean_returns = log_returns.mean()
std_returns = log_returns.std(ddof=1)

mu_avg = mean_returns.mean()
sigma_avg = std_returns.mean()
lambda_market = sigma_avg / (mu_avg + sigma_avg) if (mu_avg + sigma_avg) != 0 else 0.5

print(f"Average Market Return (mu_avg) : {mu_avg:.6f}")
print(f"Average Market Volatility (sigma) : {sigma_avg:.6f}")
print(f"Endogenous Risk Aversion (lambda) : {lambda_market:.4f}\n")

# 4. Fungsi Menghitung Payoff Matrix dan State Probabilities untuk Pasangan Aset
def calculate_payoff_and_prob(asset_L, asset_F):
    payoff_L = np.zeros((2, 2))
    payoff_F = np.zeros((2, 2))
    probs = np.zeros((2, 2))
    
    n_total = len(log_returns)
    
    # i = State Leader, j = State Follower
    for i in [0, 1]:     # 0=Up, 1=Down
        for j in [0, 1]: # 0=Up, 1=Down
            mask = (states[asset_L] == i) & (states[asset_F] == j)
            n_ij = mask.sum()
            
            # Probability (P_ij)
            p_ij = n_ij / n_total if n_total > 0 else 0
            probs[i, j] = p_ij
            
            # Subset return saat kondisi pasar berada di state i dan j
            subset_L = log_returns.loc[mask, asset_L]
            subset_F = log_returns.loc[mask, asset_F]
            
            # Hitung rata-rata dan deviasi standar harian dari subset
            # Gunakan ddof=0 agar tidak NaN jika n_ij=1, atau isikan 0 jika n_ij=0
            if n_ij > 0:
                mu_L, sig_L = subset_L.mean(), subset_L.std(ddof=0)
                mu_F, sig_F = subset_F.mean(), subset_F.std(ddof=0)
            else:
                mu_L, sig_L = 0, 0
                mu_F, sig_F = 0, 0
                
            # Utilitas Markowitz: U = (1 - lambda) * mu - lambda * sigma
            U_L = (1 - lambda_market) * mu_L - lambda_market * sig_L
            U_F = (1 - lambda_market) * mu_F - lambda_market * sig_F
            
            payoff_L[i, j] = U_L
            payoff_F[i, j] = U_F
            
    return payoff_L, payoff_F, probs

# Menghitung untuk semua kombinasi pasangan saham (Leader, Follower)
pairs = list(itertools.combinations(tickers, 2))
results = {}

for pair in pairs:
    asset_L, asset_F = pair
    pL, pF, prob = calculate_payoff_and_prob(asset_L, asset_F)
    
    # Amplitudo Probabilitas / Koefisien Kuantum a_ij = sqrt(Probability_ij)
    quantum_coeffs = np.sqrt(prob)
    
    results[pair] = {
        'Payoff_L': pL,
        'Payoff_F': pF,
        'Probabilities': prob,
        'Quantum_Coeffs': quantum_coeffs
    }
    
    print(f"--- Pasangan Leader: {asset_L} | Follower: {asset_F} ---")
    print("Matriks Payoff Leader:")
    print(np.round(pL, 6))
    print("Matriks Payoff Follower:")
    print(np.round(pF, 6))
    print("Probabilitas State (P_ij):")
    print(np.round(prob, 4))
    print("Quantum Coefficients (a_ij):")
    print(np.round(quantum_coeffs, 4))
    print("-" * 50 + "\n")


Calculating Log Returns and Market Risk Aversion...

Average Market Return (mu_avg) : -0.000090
Average Market Volatility (sigma) : 0.023353
Endogenous Risk Aversion (lambda) : 1.0039

--- Pasangan Leader: BBCA.JK | Follower: TPIA.JK ---
Matriks Payoff Leader:
[[-0.016152 -0.010167]
 [-0.007668 -0.012294]]
Matriks Payoff Follower:
[[-0.023261 -0.019645]
 [-0.016276 -0.015578]]
Probabilitas State (P_ij):
[[0.1694 0.2258]
 [0.2097 0.3952]]
Quantum Coefficients (a_ij):
[[0.4115 0.4752]
 [0.4579 0.6286]]
--------------------------------------------------

--- Pasangan Leader: BBCA.JK | Follower: ASII.JK ---
Matriks Payoff Leader:
[[-0.015068 -0.010044]
 [-0.007922 -0.011957]]
Matriks Payoff Follower:
[[-0.018265 -0.007183]
 [-0.011018 -0.01969 ]]
Probabilitas State (P_ij):
[[0.2258 0.1694]
 [0.2581 0.3468]]
Quantum Coefficients (a_ij):
[[0.4752 0.4115]
 [0.508  0.5889]]
--------------------------------------------------

--- Pasangan Leader: BBCA.JK | Follower: TLKM.JK ---
Matriks Payoff L

---

### Konstruksi Fungsi Gelombang dan Matriks Payoff

Berdasarkan metodologi **Markowitz $\times$ Game Theory (Risk Aversion Endogen)** dan **Econophysics Kuantum**, berikut adalah langkah-langkah formulasi dari data historis hingga mendapatkan fungsi gelombang $|\psi\rangle$.

#### 1. Perhitungan Return dan Penentuan State
Pertama, kita menghitung *Daily Log Return* ($R_t$) untuk mengidentifikasi pergerakan harga harian.
$$ R_t = \ln\left(\frac{P_t}{P_{t-1}}\right) $$

Setiap hari perdagangan kemudian didiskritisasi ke dalam dua *state* kuantum:
- **Naik ($|0\rangle$)**: Jika $R_t > 0$
- **Turun ($|1\rangle$)**: Jika $R_t \le 0$

#### 2. Risk Aversion Endogen ($\lambda_{\text{market}}$)
Alih-alih berasumsi menggunakan angka statis, kecenderungan penghindaran risiko (risk aversion) pasar dihitung berdasarkan volatilitas agregat:
$$ \lambda_{\text{market}} = \frac{\sigma_{avg}}{\mu_{avg} + \sigma_{avg}} $$
- $\mu_{avg}$: Rata-rata *return* seluruh aset dalam pasar/portofolio.
- $\sigma_{avg}$: Rata-rata volatilitas (standar deviasi) seluruh aset.

#### 3. Konstruksi Matriks Payoff (Utilitas Markowitz)
Misalkan kita mengamati dua aset: **Leader ($L$)** dan **Follower ($F$)**. Kita menghitung utilitas objektif Markowitz untuk masing-masing aset berdasarkan subset waktu berlakunya state gabungan $|ij\rangle$ (misal: Leader naik, Follower turun).

Fungsi utilitas untuk aset $k \in \{L, F\}$ pada state $|ij\rangle$ adalah:
$$ U_k^{(ij)} = (1 - \lambda_{\text{market}}) \cdot \mu_{k}^{(ij)} - \lambda_{\text{market}} \cdot \sigma_{k}^{(ij)} $$
- $\mu_{k}^{(ij)}$: Rata-rata return harian aset $k$ HANYA pada hari-hari di mana state $|ij\rangle$ terjadi.
- $\sigma_{k}^{(ij)}$: Standar deviasi return harian aset $k$ pada hari-hari tersebut.

Matriks Payoff untuk pasangan $(L, F)$ kemudian dapat divisualisasikan sebagai tabel interaksi (Game Theory):
| L \ F | $\ket{0}_F$ (Up) | $\ket{1}_F$ (Down) |
| :--- | :---: | :---: |
| **$\ket{0}_L$ (Up)** | $(U_L^{(00)}, U_F^{(00)})$ | $(U_L^{(01)}, U_F^{(01)})$ |
| **$\ket{1}_L$ (Down)** | $(U_L^{(10)}, U_F^{(10)})$ | $(U_L^{(11)}, U_F^{(11)})$ |

#### 4. Probabilitas State Bersama ($P_{ij}$)
Kita menghitung seberapa sering gabungan pergerakan aset ini terjadi. Jika total hari pengamatan adalah $N$ dan gabungan state $|ij\rangle$ terjadi sebanyak $n_{ij}$ kali, maka probabilitas empirisnya:
$$ P_{ij} = \frac{n_{ij}}{N} $$
*(Catatan: $P_{00} + P_{01} + P_{10} + P_{11} = 1$)*

#### 5. Fungsi Gelombang Kuantum ($|\psi\rangle$)
Terakhir, kita merepresentasikan probabilitas bersama ini ke dalam formalisme mekanika kuantum. Amplitudo probabilitas (*Quantum Coefficient*), $a_{ij}$, adalah akar kuadrat dari probabilitas empiris:
$$ a_{ij} = \sqrt{P_{ij}} $$

Sehingga fungsi gelombang sistem dua aset tersebut (Leader-Follower) diekspresikan sebagai superposisi linear dari semua kemungkinan state:
$$ |\psi\rangle = a_{00} |00\rangle + a_{01} |01\rangle + a_{10} |10\rangle + a_{11} |11\rangle $$

Fungsi gelombang $|\psi\rangle$ inilah yang nantinya kita observasi untuk mencari *Quantum Mutual Information* (seberapa kuat keterikatan/entanglement antara aset L dan F) serta matriks densitas $\rho = |\psi\rangle\langle\psi|$.


In [4]:
import numpy as np
import itertools
import pandas as pd
import math

print("1. Menghitung Log Return Harian dan State (0=Naik, 1=Turun)...")
# Dari data 'df' sebelumnya, kita perlu pastikan kita punya seluruh ticker.
# Kita akan mengambil ulang agar lebih bersih dan memiliki dataframe harga yang utuh
data = yf.download(tickers, period=period, interval=interval, progress=False)

if isinstance(data.columns, pd.MultiIndex):
    prices = data['Close']
else:
    prices = data['Close']  # backup

prices = prices.dropna()

# R_t = ln(P_t / P_{t-1})
log_returns = np.log(prices / prices.shift(1)).dropna()

# Diskritisasi: 0 jika R_t > 0, 1 jika R_t <= 0
states = (log_returns <= 0).astype(int)

print("\n2. Menghitung Risk Aversion Endogen (Lambda Market) ...")
mu_avg = log_returns.mean().mean()     # Rata-rata return seluruh pasar
sigma_avg = log_returns.std(ddof=1).mean()  # Rata-rata volatilitas seluruh pasar

lambda_market = sigma_avg / (mu_avg + sigma_avg) if (mu_avg + sigma_avg) != 0 else 0.5
print(f"   - Mu_avg    : {mu_avg:.6f}")
print(f"   - Sigma_avg : {sigma_avg:.6f}")
print(f"   - Lambda    : {lambda_market:.6f}")

print("\n3 & 4 & 5. Konstruksi Matriks Payoff, Probabilitas, dan Fungsi Gelombang...")
def calculate_quantum_payoff(asset_L, asset_F):
    payoff_L = np.zeros((2, 2))
    payoff_F = np.zeros((2, 2))
    probs = np.zeros((2, 2))
    
    n_total = len(log_returns)
    
    for i in [0, 1]:     # State Leader
        for j in [0, 1]: # State Follower
            # Filter data di mana state L=i dan state F=j
            mask = (states[asset_L] == i) & (states[asset_F] == j)
            n_ij = mask.sum()
            
            # Probabilitas Empiris (P_ij)
            p_ij = n_ij / n_total if n_total > 0 else 0
            probs[i, j] = p_ij
            
            # Ambil subset return
            subset_L = log_returns.loc[mask, asset_L]
            subset_F = log_returns.loc[mask, asset_F]
            
            # Hitung rata-rata dan std subset
            if n_ij > 0:
                mu_L, sig_L = subset_L.mean(), subset_L.std(ddof=0)
                mu_F, sig_F = subset_F.mean(), subset_F.std(ddof=0)
            else:
                mu_L, sig_L = 0, 0
                mu_F, sig_F = 0, 0
                
            # Utilitas Markowitz (Payoff)
            payoff_L[i, j] = (1 - lambda_market) * mu_L - lambda_market * sig_L
            payoff_F[i, j] = (1 - lambda_market) * mu_F - lambda_market * sig_F
            
    return payoff_L, payoff_F, probs

pairs = list(itertools.permutations(tickers, 2))
quantum_results = {}

for pair in pairs:
    asset_L, asset_F = pair
    pL, pF, prob = calculate_quantum_payoff(asset_L, asset_F)
    
    # Amplitudo Probabilitas / Koefisien Kuantum a_ij = sqrt(Probability_ij)
    quantum_coeffs = np.sqrt(prob)
    
    quantum_results[pair] = {
        'Payoff_L': pL,
        'Payoff_F': pF,
        'Probabilities': prob,
        'Quantum_Coeffs': quantum_coeffs
    }
    
    print(f"\n{'='*50}")
    print(f"Pasangan Leader: {asset_L} | Follower: {asset_F}")
    print(f"{'='*50}")
    print("Matriks Payoff Leader (U_L):")
    print(np.round(pL, 6))
    print("\nMatriks Payoff Follower (U_F):")
    print(np.round(pF, 6))
    print("\nQuantum Coefficients (a_00, a_01, a_10, a_11):")
    print(np.round(quantum_coeffs, 4))
    print(f"\nFungsi Gelombang |psi> : ")
    print(f"{quantum_coeffs[0,0]:.4f} |00> + {quantum_coeffs[0,1]:.4f} |01> + "
          f"{quantum_coeffs[1,0]:.4f} |10> + {quantum_coeffs[1,1]:.4f} |11>")


1. Menghitung Log Return Harian dan State (0=Naik, 1=Turun)...

2. Menghitung Risk Aversion Endogen (Lambda Market) ...
   - Mu_avg    : -0.000141
   - Sigma_avg : 0.023395
   - Lambda    : 1.006070

3 & 4 & 5. Konstruksi Matriks Payoff, Probabilitas, dan Fungsi Gelombang...

Pasangan Leader: BBCA.JK | Follower: TPIA.JK
Matriks Payoff Leader (U_L):
[[-0.016225 -0.010221]
 [-0.007665 -0.012302]]

Matriks Payoff Follower (U_F):
[[-0.023366 -0.019655]
 [-0.016356 -0.01574 ]]

Quantum Coefficients (a_00, a_01, a_10, a_11):
[[0.4115 0.4752]
 [0.4579 0.6286]]

Fungsi Gelombang |psi> : 
0.4115 |00> + 0.4752 |01> + 0.4579 |10> + 0.6286 |11>

Pasangan Leader: BBCA.JK | Follower: ASII.JK
Matriks Payoff Leader (U_L):
[[-0.015135 -0.0101  ]
 [-0.007922 -0.011951]]

Matriks Payoff Follower (U_F):
[[-0.018354 -0.007174]
 [-0.011077 -0.019741]]

Quantum Coefficients (a_00, a_01, a_10, a_11):
[[0.4752 0.4115]
 [0.508  0.5889]]

Fungsi Gelombang |psi> : 
0.4752 |00> + 0.4115 |01> + 0.5080 |10> + 0.5889

### Konstruksi Matriks Densitas & Quantum Mutual Information (QMI)

Berdasarkan fungsi gelombang $\ket{psi}$ yang didapat pada langkah sebelumnya, kita sekarang akan menghitung derajat keterikatan (*entanglement*) informasi antara pergerakan aset Leader ($L$) dan Follower ($F$). 

Dalam mekanika kuantum makroskopis (Econophysics), relasi ini tidak diukur dengan korelasi Pearson biasa, melainkan dengan **Quantum Mutual Information (QMI)** atau $J_{ij}$ yang diekstrak dari **Von Neumann Entropy**.

#### 1. Membentuk Matriks Densitas Sistem ($\rho$)
Dari amplitudo probabilitas $a_{ij}$ sebelumnya, fungsi gelombang dua aset dinyatakan sebagai vektor berdimensi $4 	imes 1$:
$$ \ket{\psi}= \begin{bmatrix} a_{00} \ a_{01} \ a_{10} \ a_{11} \end{bmatrix} $$

Matriks densitas murni untuk keseluruhan sistem adalah matriks $4 	imes 4$, dihitung melalui perkalian *outer product* (ket-bra):
$$ \rho = \ket{\psi} \bra{\psi} = \begin{bmatrix}
a_{00}a_{00}^* & a_{00}a_{01}^* & a_{00}a_{10}^* & a_{00}a_{11}^* \\
a_{01}a_{00}^* & a_{01}a_{01}^* & a_{01}a_{10}^* & a_{01}a_{11}^* \\
a_{10}a_{00}^* & a_{10}a_{01}^* & a_{10}a_{10}^* & a_{10}a_{11}^* \\
a_{11}a_{00}^* & a_{11}a_{01}^* & a_{11}a_{10}^* & a_{11}a_{11}^*
\end{bmatrix} $$
*(Karena nilai kita adalah *real probabilities*, nilai $a^*$ konjugat kompleks sama dengan $a$ itu sendiri)*.

#### 2. Menghitung Reduced Density Matrix dengan Partial Trace ($\rho_L$ dan $\rho_F$)
Karena kita ingin mengetahui informasi individual masing-masing aset ketika dipisahkan dari pasangannya, kita melakukan operasi *Partial Trace* terhadap matriks Densitas $\rho$.

- **Untuk Aset Leader (L):** Melakukan *trace out* sub-sistem F. Kita menjumlahkan elemen diagonal blok-blok matriks.
  $$ \rho_L = \text{Tr}_F(\rho) = \begin{bmatrix} a_{00}^2 + a_{01}^2 & a_{00}a_{10} + a_{01}a_{11} \\ a_{10}a_{00} + a_{11}a_{01} & a_{10}^2 + a_{11}^2 \end{bmatrix} $$

- **Untuk Aset Follower (F):** Melakukan *trace out* sub-sistem L. Kita menjumlahkan elemen di dalam setiap blok $2 \times 2$.
  $$ \rho_F = \text{Tr}_L(\rho) = \begin{bmatrix} a_{00}^2 + a_{10}^2 & a_{00}a_{01} + a_{10}a_{11} \\ a_{01}a_{00} + a_{11}a_{10} & a_{01}^2 + a_{11}^2 \end{bmatrix} $$

#### 3. Menghitung Entropi Von Neumann ($S(\rho)$)
Von Neumann Entropy dari suatu matriks densitas $\sigma$ adalah ukuran ketidakpastian kuantum:
$$ S(\sigma) = -\text{Tr}(\sigma \ln \sigma) $$

Karena menghitung logaritma dari matriks bisa rumit, secara komputasi (Numerik) kita biasanya:
1. Mencari nilai *eigenvalues* ($\lambda_k$) dari matriks densitas $\sigma$.
2. Menghitung entropinya lewat jumlah nilai eigen: $S(\sigma) = -\sum_{k} \lambda_k \ln(\lambda_k)$.

Kita menghitung tiga entropi secara terpisah: 
- $S(\rho_L)$ : Entropi individual Leader.
- $S(\rho_F)$ : Entropi individual Follower.
- $S(\rho_{LF})$ : Entropi gabungan (*Joint entropy*). *Catatan: Untuk keadaan murni (pure state) seperti awal kita, $S(\rho_{LF})$ selalu $0$.*

#### 4. Menghasilkan Konstanta Interaksi Hamiltonian (Quantum Mutual Information / $J_{LF}$)
Derajat ketergantungan sejati (*entanglement*) antar kedua aset akhirnya dirumuskan sebagai QMI. Semakin besar nilainya, semakin tebal "ikatan" antar aset di pasar finansial.
$$ I(L : F) = S(\rho_L) + S(\rho_F) - S(\rho_{LF}) $$

Pada ekuasi Ising Hamiltonian final, kita menetapkan parameter interaksi strategis ($J_{LF}$) sama dengan QMI tersebut (bila interaksinya searah/koordinasi).
$$ J_{LF} = I(L:F) $$


In [5]:
import numpy as np
from scipy.linalg import eigh

print("6. Menghitung Reduced Density Matrix, Entropi Von Neumann, dan Quantum Mutual Information\n")

def vn_entropy(rho):
    # Menghitung Von Neumann Entropy: -Tr(rho * ln(rho))
    # Menggunakan dekomposisi nilai eigen karena matriks Hermite/Simetris
    evals = eigh(rho)[0] # eigh untuk matriks Hermite mengembalikan eigenvalues array terurut
    
    # Toleransi untuk membuang eigen value 0 (karena 0 * ln(0) limitnya adalah 0)
    evals = np.clip(evals, 1e-12, 1.0)
    
    entropy_val = -np.sum(evals * np.log(evals))
    return entropy_val

# Dictionary untuk menyimpan nilai interaksi J_ij (QMI)
interaction_J = {}

for pair, res in quantum_results.items():
    asset_L, asset_F = pair
    a = res['Quantum_Coeffs'] # Matriks 2x2 amplitudes
    
    # 1. State Vector (4x1)
    # Rata kanan (flatten) mengikuti urutan: |00>, |01>, |10>, |11>
    psi = np.array([a[0,0], a[0,1], a[1,0], a[1,1]])
    
    # 2. Matriks Densitas Sistem Keseluruhan (4x4)
    # rho = |psi><psi|
    rho_system = np.outer(psi, psi)
    
    # 3. Partial Trace untuk Rho_Leader (2x2)
    # Melacak keluar atribut (Trace Out) Follower
    rho_L = np.array([
        [psi[0]*psi[0] + psi[1]*psi[1], psi[0]*psi[2] + psi[1]*psi[3]],
        [psi[2]*psi[0] + psi[3]*psi[1], psi[2]*psi[2] + psi[3]*psi[3]]
    ])
    
    # Partial Trace untuk Rho_Follower (2x2)
    # Melacak keluar atribut (Trace Out) Leader
    rho_F = np.array([
        [psi[0]*psi[0] + psi[2]*psi[2], psi[0]*psi[1] + psi[2]*psi[3]],
        [psi[1]*psi[0] + psi[3]*psi[2], psi[1]*psi[1] + psi[3]*psi[3]]
    ])
    
    # 4. Menghitung Entropi Sistem secara terpisah
    S_L = vn_entropy(rho_L)
    S_F = vn_entropy(rho_F)
    
    # Joint entropy dari pure state (rho_system) harusnya hampir mendekati ~ 0 secara teoritis 
    # Namun kita tetap hitung untuk generalisasi jika ada mixed state
    S_system = vn_entropy(rho_system)
    
    # 5. Quantum Mutual Information
    I_LF = S_L + S_F - S_system
    
    # Nilai QMI ini akan kita pakai sebagai Kekuatan Interaksi J (Ising Hamiltonian)
    interaction_J[pair] = I_LF
    
    print(f"--- QMI untuk Pasangan [{asset_L} dan {asset_F}] ---")
    print(f"Entropi Von Neumann {asset_L} (S_L) : {S_L:.4f}")
    print(f"Entropi Von Neumann {asset_F} (S_F) : {S_F:.4f}")
    print(f"Joint Entropy Sistem (S_LF)       : {S_system:.4f}")
    print(f"Quantum Mutual Info / J_LF (I)    : {I_LF:.4f}\n")


6. Menghitung Reduced Density Matrix, Entropi Von Neumann, dan Quantum Mutual Information

--- QMI untuk Pasangan [BBCA.JK dan TPIA.JK] ---
Entropi Von Neumann BBCA.JK (S_L) : 0.0125
Entropi Von Neumann TPIA.JK (S_F) : 0.0125
Joint Entropy Sistem (S_LF)       : 0.0000
Quantum Mutual Info / J_LF (I)    : 0.0250

--- QMI untuk Pasangan [BBCA.JK dan ASII.JK] ---
Entropi Von Neumann BBCA.JK (S_L) : 0.0317
Entropi Von Neumann ASII.JK (S_F) : 0.0317
Joint Entropy Sistem (S_LF)       : 0.0000
Quantum Mutual Info / J_LF (I)    : 0.0633

--- QMI untuk Pasangan [BBCA.JK dan TLKM.JK] ---
Entropi Von Neumann BBCA.JK (S_L) : 0.0393
Entropi Von Neumann TLKM.JK (S_F) : 0.0393
Joint Entropy Sistem (S_LF)       : 0.0000
Quantum Mutual Info / J_LF (I)    : 0.0787

--- QMI untuk Pasangan [TPIA.JK dan BBCA.JK] ---
Entropi Von Neumann TPIA.JK (S_L) : 0.0125
Entropi Von Neumann BBCA.JK (S_F) : 0.0125
Joint Entropy Sistem (S_LF)       : 0.0000
Quantum Mutual Info / J_LF (I)    : 0.0250

--- QMI untuk Pasanga

### Contoh Kalkulasi Manual: Quantum Mutual Information (QMI)

Melanjutkan ilustrasi sebelumnya... Misalkan berdasarkan data probabilitas empiris $P_{ij}$, kita dapati fungsi gelombang $|\psi\rangle$ (vektor $4 \times 1$) untuk pasangan **BBCA** (Leader) dan **ASII** (Follower) memiliki amplitudo probabilitas:
- $a_{00} = \sqrt{0.40} \approx 0.632$
- $a_{01} = \sqrt{0.10} \approx 0.316$
- $a_{10} = \sqrt{0.20} \approx 0.447$
- $a_{11} = \sqrt{0.30} \approx 0.548$

*(Catatan: $a_{00}^2 + a_{01}^2 + a_{10}^2 + a_{11}^2 = 0.4 + 0.1 + 0.2 + 0.3 = 1.0$)*

#### 1. Matriks Densitas Keseluruhan ($\rho$)
Outer product dari vektor $|\psi\rangle$:
$$ \rho = \begin{bmatrix} 0.632 \\ 0.316 \\ 0.447 \\ 0.548 \end{bmatrix} \begin{bmatrix} 0.632 & 0.316 & 0.447 & 0.548 \end{bmatrix} = \begin{bmatrix} 0.400 & 0.200 & 0.283 & 0.346 \\ 0.200 & 0.100 & 0.141 & 0.173 \\ 0.283 & 0.141 & 0.200 & 0.245 \\ 0.346 & 0.173 & 0.245 & 0.300 \end{bmatrix} $$

Karena keadaan simulasi kita berupa *pure state*, $\rho$ memiliki satu nilai *eigenvalue* bernilai $1$ dan sisanya $0$. Sehingga entropi gabungan otomatis minimum: **$S(\rho_{LF}) = 0$**.

#### 2. Partial Trace untuk Reduced Density Matrix
**Untuk Leader (BBCA) $\rightarrow \rho_L$**:
Menjumlahkan silang blok dari subsystem F:
$$ \rho_L = \begin{bmatrix} (a_{00}^2 + a_{01}^2) & (a_{00}a_{10} + a_{01}a_{11}) \\ (a_{10}a_{00} + a_{11}a_{01}) & (a_{10}^2 + a_{11}^2) \end{bmatrix} $$
$$ \rho_L = \begin{bmatrix} (0.40 + 0.10) & (0.283 + 0.173) \\ (0.283 + 0.173) & (0.20 + 0.30) \end{bmatrix} = \begin{bmatrix} 0.500 & 0.456 \\ 0.456 & 0.500 \end{bmatrix} $$

**Untuk Follower (ASII) $\rightarrow \rho_F$**:
Mengambil batas *trace* (penjumlahan memusat diagonal) dari pengaruh Leader:
$$ \rho_F = \begin{bmatrix} (a_{00}^2 + a_{10}^2) & (a_{00}a_{01} + a_{10}a_{11}) \\ (a_{01}a_{00} + a_{11}a_{10}) & (a_{01}^2 + a_{11}^2) \end{bmatrix} $$
$$ \rho_F = \begin{bmatrix} (0.40 + 0.20) & (0.200 + 0.245) \\ (0.200 + 0.245) & (0.10 + 0.30) \end{bmatrix} = \begin{bmatrix} 0.600 & 0.445 \\ 0.445 & 0.400 \end{bmatrix} $$

#### 3. Menghitung Von Neumann Entropy ($S$)
Kita mencari *eigenvalues* ($\lambda$) dari matriks $2 \times 2$ tersebut secara aljabar linier/numerik, kemudian menghitung $S = -\sum \lambda \ln(\lambda)$.

Misalkan secara numerik diketemukan *eigenvalue* dari $\rho_L$ adalah $\lambda_1 \approx 0.956$ dan $\lambda_2 \approx 0.044$.
Maka entropinya:
$$ S(\rho_L) = -(0.956 \ln 0.956 + 0.044 \ln 0.044) \approx 0.183 $$

Misalkan *eigenvalue* dari $\rho_F$ secara serupa menghasilkan entropi:
$$ S(\rho_F) \approx 0.231 $$

#### 4. Mencari Konstanta Interaksi Hamiltonian/QMI ($J$)
Kekuatan iteraksi jaringan (*Quantum Entanglement*) antar-aset strategi BBCA dan ASII merupakan selisih entropi marginal berbanding gabungan:
$$ J = I(L:F) = S(\rho_L) + S(\rho_F) - S(\rho_{LF}) $$
$$ J = 0.183 + 0.231 - 0 = 0.414 $$

Nilai $J = 0.414$ ini merepresentasikan ikatan non-linear antar saham yang nantinya akan dimasukkan langsung ke pembentukan tensor dari model **Ising Hamiltonian**.


---

### Konstruksi Parameter Bias ($h$) untuk Ising Hamiltonian

Setelah mendapatkan nilai kekuatan interaksi ($J_{ij}$) dari Quantum Mutual Information, langkah berikutnya menuju penyusunan model **Ising Hamiltonian** ($H$) adalah menentukan **Parameter Bias ($h_i$)** untuk masing-masing aset (Leader dan Follower).

Dalam mekanika statistik dan komputasi kuantum, $h_i$ merepresentasikan *medan magnet eksternal* (external magnetic field) yang mempengaruhi kecenderungan (probabilitas intrinsik) suatu qubit/aset untuk berada di state $|0\rangle$ (Naik) atau state $|1\rangle$ (Turun) terlepas dari pergerakan aset pasangannya.

#### Menghitung Bias dari Matriks Payoff
Nilai Bias dihitung dari **selisih total ekspektasi utilitas (Payoff) ketika aset tersebut naik dibandingkan ketika ia turun**.

Misalkan kita kembali pada matriks Payoff Markowitz $2 \times 2$:

**Untuk Aset Leader (L):**
Kecenderungan Leader untuk naik (state $|0\rangle$) dipengaruhi oleh payoff-nya saat naik terlepas dari situasi Follower.
$$ \text{Expected Payoff } L \text{ Naik} = U_L^{(00)} + U_L^{(01)} $$
Kecenderungan Leader untuk turun (state $|1\rangle$) dipengaruhi oleh payoff-nya saat turun.
$$ \text{Expected Payoff } L \text{ Turun} = U_L^{(10)} + U_L^{(11)} $$
Sehingga Bias untuk Leader:
$$ h_L = \frac{\text{Expected Payoff } L \text{ Naik} - \text{Expected Payoff } L \text{ Turun}}{2} $$
$$ h_L = \frac{(U_L^{(00)} + U_L^{(01)}) - (U_L^{(10)} + U_L^{(11)})}{2} $$

**Untuk Aset Follower (F):**
Serupa dengan Leader, kecenderungan Follower dihitung dengan membandingkan kolom matriks (karena state Follower direpresentasikan oleh posisi kolom: $|0\rangle$ di kolom pertama, $|1\rangle$ di kolom kedua).
$$ \text{Expected Payoff } F \text{ Naik} = U_F^{(00)} + U_F^{(10)} $$
$$ \text{Expected Payoff } F \text{ Turun} = U_F^{(01)} + U_F^{(11)} $$
Sehingga Bias untuk Follower:
$$ h_F = \frac{\text{Expected Payoff } F \text{ Naik} - \text{Expected Payoff } F \text{ Turun}}{2} $$
$$ h_F = \frac{(U_F^{(00)} + U_F^{(10)}) - (U_F^{(01)} + U_F^{(11)})}{2} $$

#### Makna Nilai $h_i$
- Jika **$h_i > 0$**, ini secara dominan mengindikasikan struktur utilitas mendukung aset $i$ untuk berada di kondisi Uptrend/Naik ($|0\rangle$).
- Jika **$h_i < 0$**, utilitas secara dominan mendikte aset $i$ untuk berada di kondisi Downtrend/Turun ($|1\rangle$).

#### 1. Persiapan Ising Hamiltonian VQE
Nantinya, kedua parameter Hamiltonian ini (Bias individual $h_i$ dan Interaksi mutual $J_{ij}$) akan kita susun ke dalam operator matriks Ising 2-qubit untuk dioptimasi menggunakan algoritma kuantum iteratif **VQE (Variational Quantum Eigensolver)**:
$$ H_{LF} = -h_L \sigma_L^z - h_F \sigma_F^z - J_{LF} (\sigma_L^z \otimes \sigma_F^z) $$
*Note: Nilai Hamiltonian di atas baru untuk 1 pasangan (2 Qubit). Untuk portofolio utuh, kita akan mensintesis matriks $H$ raksasa berukuran $4\times 4$ untuk ke-4 saham.*


In [6]:
print("7. Menghitung Parameter Bias (h) dari Matriks Payoff\n")

# Dictionary untuk menyimpan nilai Bias individual h_i 
bias_h = {ticker: [] for ticker in tickers}

for pair, res in quantum_results.items():
    asset_L, asset_F = pair
    pL = res['Payoff_L']
    pF = res['Payoff_F']
    
    # --- Bias Leader (L) ---
    # Ekspektasi saat Leader Naik (Baris 0) vs Turun (Baris 1)
    U_L_Up = pL[0, 0] + pL[0, 1]
    U_L_Down = pL[1, 0] + pL[1, 1]
    h_L = (U_L_Up - U_L_Down) / 2
    
    # --- Bias Follower (F) ---
    # Ekspektasi saat Follower Naik (Kolom 0) vs Turun (Kolom 1)
    U_F_Up = pF[0, 0] + pF[1, 0]
    U_F_Down = pF[0, 1] + pF[1, 1]
    h_F = (U_F_Up - U_F_Down) / 2
    
    # Karena satu saham bisa dipasangkan dengan beberapa saham lain (sebagai L maupun F), 
    # di dalam Network Theory dan Hamiltonian kita ingin mengambil Rata-Rata Bias secara makroskopis
    bias_h[asset_L].append(h_L)
    bias_h[asset_F].append(h_F)
    
    print(f"--- Bias untuk Pasangan [{asset_L} dan {asset_F}] ---")
    print(f"Bias {asset_L} (h_L) : {h_L:.6f}")
    print(f"Bias {asset_F} (h_F) : {h_F:.6f}\n")

print("--- Rata-Rata Bias Keseluruhan untuk Masing-Masing Qubit (Saham) ---")
final_bias = {}
for ticker, h_values in bias_h.items():
    if len(h_values) > 0:
        avg_h = np.mean(h_values)
        final_bias[ticker] = avg_h
        trend = "Cenderung Naik (Uptrend)" if avg_h > 0 else "Cenderung Turun (Downtrend)"
        print(f"Saham {ticker:8s} | h_i = {avg_h:>10.6f}  => {trend}")


7. Menghitung Parameter Bias (h) dari Matriks Payoff

--- Bias untuk Pasangan [BBCA.JK dan TPIA.JK] ---
Bias BBCA.JK (h_L) : -0.003239
Bias TPIA.JK (h_F) : -0.002163

--- Bias untuk Pasangan [BBCA.JK dan ASII.JK] ---
Bias BBCA.JK (h_L) : -0.002681
Bias ASII.JK (h_F) : -0.001257

--- Bias untuk Pasangan [BBCA.JK dan TLKM.JK] ---
Bias BBCA.JK (h_L) : -0.002710
Bias TLKM.JK (h_F) : -0.001646

--- Bias untuk Pasangan [TPIA.JK dan BBCA.JK] ---
Bias TPIA.JK (h_L) : -0.002163
Bias BBCA.JK (h_F) : -0.003239

--- Bias untuk Pasangan [TPIA.JK dan ASII.JK] ---
Bias TPIA.JK (h_L) : -0.001830
Bias ASII.JK (h_F) : 0.003079

--- Bias untuk Pasangan [TPIA.JK dan TLKM.JK] ---
Bias TPIA.JK (h_L) : -0.003654
Bias TLKM.JK (h_F) : -0.003202

--- Bias untuk Pasangan [ASII.JK dan BBCA.JK] ---
Bias ASII.JK (h_L) : -0.001257
Bias BBCA.JK (h_F) : -0.002681

--- Bias untuk Pasangan [ASII.JK dan TPIA.JK] ---
Bias ASII.JK (h_L) : 0.003079
Bias TPIA.JK (h_F) : -0.001830

--- Bias untuk Pasangan [ASII.JK dan TLKM.JK